# Chapitre 1.1 : ce qu’est vraiment un modèle de langage

Une IA qui écrit du texte ne fait qu’une chose : deviner le caractère suivant, puis recommencer. Dans ce carnet, tu construis le plus petit modèle de langage qui existe, dans un vrai petit programme : il compte, dans un texte, ce qui suit chaque groupe de caractères, et s’en sert pour deviner la suite. Il apprend sur le texte du site lui-même.

Ce carnet accompagne [la page du chapitre 1.1](https://chipat-neko.github.io/forge-ia/partie-1/1-1/) : tu peux la lire avant, ou en même temps.

**Ce que tu vas faire**

1. compter ce qui suit un caractère ;
2. en tirer des probabilités, et comprendre le lissage ;
3. tirer au sort le caractère suivant, avec la température ;
4. donner plus de contexte au modèle, et voir ce que ça change ;
5. mesurer sa surprise sur un texte qu’il n’a jamais lu : la perplexité ;
6. relever un défi, puis vérifier que ce carnet écrit exactement les mêmes textes que la page du site.

**Comment l’utiliser**

Il n’y a rien à installer. Lance les cellules de code de haut en bas : clique sur le bouton rond avec un triangle, à gauche de la cellule, ou tape Maj + Entrée. Tu peux aussi choisir « Exécution », puis « Tout exécuter ». Tu peux modifier le code et le relancer autant de fois que tu veux : tu ne peux rien casser.

## Étape 0 : récupérer le code

Le code du projet est dans un dépôt GitHub public. La cellule suivante le télécharge avec `git clone`, puis indique à Python où trouver le projet. Le projet n’utilise que ce qui est livré avec Python : il n’y a rien d’autre à installer.

Elle définit aussi deux petites fonctions : `nombre`, qui écrit les nombres à la française, avec une virgule, et `voir`, qui écrit un caractère de façon lisible : un espace s’y voit comme « espace ».

In [ ]:
import os
import sys
from pathlib import Path

# Sur Colab, on télécharge le dépôt public. Quand ce carnet est lancé par les tests
# du projet, la variable FORGE_IA_LOCAL donne le chemin d’un dépôt déjà présent :
# on l’utilise, sans rien télécharger.
depot_local = os.environ.get("FORGE_IA_LOCAL")
if depot_local:
    depot = Path(depot_local)
else:
    depot = Path("forge-ia")
    if not depot.exists():
        !git clone --depth 1 https://github.com/chipat-neko/forge-ia

projet = depot / "projets" / "1.1-modele-de-langage"
if not projet.exists():
    raise FileNotFoundError("Le dépôt n’a pas pu être téléchargé. Vérifie ta connexion, puis relance cette cellule.")

# Python cherche les modules dans les dossiers de sys.path : on y ajoute le projet.
sys.path.insert(0, str(projet.resolve()))


def nombre(x, decimales=None):
    # Un nombre à la française. « :, » sépare les milliers par des virgules, à l’anglaise :
    # on les remplace par l’espace fine (\u202f), puis le point par une virgule.
    # Sans nombre de décimales, juste ce qu’il faut (6 au plus) : 0,1 et pas 0,100000.
    x = x + 0.0  # -0.0 + 0.0 donne 0.0 : jamais « -0 »
    texte = f"{x:,.{6 if decimales is None else decimales}f}"
    if decimales is None:
        texte = texte.rstrip("0").rstrip(".")
    return texte.replace(",", "\u202f").replace(".", ",")


def voir(caractere):
    # Un espace ou un retour à la ligne ne se voient pas : on les écrit en toutes lettres.
    noms = {" ": "espace", "\n": "retour à la ligne"}
    return noms.get(caractere, f"«\u202f{caractere}\u202f»")


print("Le code du projet est prêt.")

## Étape 1 : compter ce qui suit un caractère

Un modèle de langage devine la suite d’un texte. Le plus simple se contente de compter. On lui donne un texte à lire, le **texte d’apprentissage**, et il note, pour chaque caractère, ceux qui viennent juste après lui, et combien de fois.

Ici, le texte est la prose du site : les chapitres 0.1 à 0.5 et 1.3. On en a retiré les blocs de code, les balises, les calculs, et les marques de mise en forme : le gras, les `#` des titres et les adresses des liens (leur texte reste). Dans le code en ligne, seuls les accents graves sont retirés : son contenu reste tel quel. On garde les 90 premiers pour cent pour apprendre, et les 10 derniers pour vérifier, à l’étape 5, ce que le modèle devine d’un texte qu’il n’a jamais lu.

La fonction `compter(texte, n)` fait ce travail pour un **contexte** de `n` caractères : le groupe de `n` caractères qui précède. Pour `n = 1`, le contexte est le caractère juste avant. Elle rend un dictionnaire : à chaque contexte, le compte de ce qui le suit.

In [ ]:
from modele_de_langage.ngrammes import alphabet, charger_texte, compter, couper

texte = charger_texte()
lettres = alphabet(texte)  # les caractères différents du texte, du plus petit point de code au plus grand
entrainement, validation = couper(texte)

print(f"Le texte du site fait {nombre(len(texte))} caractères, dont {len(lettres)} différents.")
print(f"{nombre(len(entrainement))} pour apprendre, {nombre(len(validation))} pour vérifier.")
print()
print(texte[:200])
print()

comptes = compter(entrainement, 1)
suivants = comptes["q"]
print(f"Après « q » (le contexte est un caractère), le texte met\u202f:")
for caractere, fois in suivants.most_common(3):
    print(f"  {voir(caractere)}, {nombre(fois)} fois")

suivants = comptes[" "]
print("Après un espace, les cinq plus fréquents\u202f:")
for caractere, fois in suivants.most_common(5):
    print(f"  {voir(caractere)}, {nombre(fois)} fois")

## Étape 2 : des comptes aux probabilités

Les comptes disent ce qui est arrivé. Pour deviner, il faut des **probabilités** : la part de chaque caractère parmi tous ceux qui ont suivi ce contexte. Si « e » a suivi 30 fois sur 100, sa probabilité est 0,3. On divise chaque compte par le total.

In [ ]:
for contexte in ["q", "e"]:
    suivants = comptes[contexte]
    total = sum(suivants.values())
    print(f"Après « {contexte} », {nombre(total)} caractères ont suivi\u202f:")
    for caractere, fois in suivants.most_common(3):
        print(f"  {voir(caractere)}\u202f: {nombre(fois)} sur {nombre(total)}, probabilité {nombre(fois / total, 3)}")

## Étape 3 : tirer au sort, avec la température

Pour écrire, le modèle ne prend pas toujours le caractère le plus probable : il tourne en rond. Il **tire au sort**, selon les probabilités : un caractère à 30 % sort trois fois sur dix. On pose les poids de tous les caractères bout à bout, et un nombre au hasard entre 0 et 1 en désigne un. Le hasard d’un ordinateur est un calcul : une **graine**, un entier, donne une suite de nombres qui a l’air prise au hasard, et toujours la même. C’est le même `Hasard` que celui de la page du site.

On ne tire que parmi les caractères déjà vus après ce contexte : les autres ont un poids nul. Et si le modèle n’a jamais vu les `n` derniers caractères, il **recule** : il regarde les `n − 1` derniers, puis `n − 2`, jusqu’à 0 caractère, qu’il connaît toujours.

La **température** règle l’audace du tirage. À 1, le modèle suit ses comptes. À 0,5, il durcit : le plus compté l’emporte encore plus souvent. À 2, il adoucit : les caractères rares sortent plus souvent. À 0, il ne tire plus : il prend toujours le plus compté.

La cellule suivante écrit 120 caractères à chaque température, avec trois caractères de contexte. Change la graine, le départ ou la température, puis relance.

In [ ]:
from modele_de_langage.ngrammes import compter_par_ordre, generer

n = 3
comptes_par_ordre = compter_par_ordre(entrainement, n)  # les comptes des ordres 0, 1, 2 et 3
depart = "Le "
graine = 0
for temperature in [0, 0.5, 1, 2]:
    suite = generer(comptes_par_ordre, lettres, n, depart, 120, temperature, graine)
    print(f"Température {nombre(temperature)}\u202f:")
    print(suite.replace("\n", " "))
    print()

À température 0, le modèle choisit toujours le plus compté : le texte est lisible, mais dès qu’un groupe de caractères revient, la suite se répète sans fin. La fonction `tourne_en_rond` le détecte : elle regarde si les 20 derniers caractères sont déjà apparus plus tôt. Aux autres températures, le tirage au sort fait varier le chemin, et la suite se répète beaucoup moins.

In [ ]:
from modele_de_langage.ngrammes import tourne_en_rond

for temperature in [0, 1]:
    suite = generer(comptes_par_ordre, lettres, 3, "Le ", 300, temperature, 0)
    reponse = "oui" if tourne_en_rond(suite) else "non"
    print(f"À la température {nombre(temperature)}, le texte de 300 caractères tourne en rond\u202f: {reponse}.")

## Étape 4 : plus de contexte

Jusqu’ici, le modèle regardait 3 caractères. Plus il en regarde, plus il a de quoi deviner : avec un seul caractère, il ne sait presque rien de ce qui suit ; avec six, il reconnaît des mots et des expressions entières du texte d’apprentissage. C’est la même idée que dans un grand modèle : donner plus de contexte, c’est mieux deviner.

La cellule suivante écrit 100 caractères pour chaque `n` de 1 à 6, après le départ « de la carte », à température 1. Regarde comment le texte change : avec peu de contexte, des syllabes presque françaises ; avec beaucoup, des morceaux entiers du texte du site.

In [ ]:
for n in range(1, 7):
    comptes_n = compter_par_ordre(entrainement, n)
    suite = generer(comptes_n, lettres, n, "de la carte ", 100, 1, 0)
    print(f"n = {n} ({nombre(len(comptes_n[n]))} contextes différents)\u202f:")
    print(suite[len("de la carte "):].replace("\n", " "))
    print()

Plus de contexte a un prix : le nombre de contextes différents grandit vite, et chacun est vu de moins en moins souvent. Avec 6 caractères, beaucoup de contextes n’ont été vus qu’une fois : le modèle y recopie ce qui a suivi, sans rien deviner. Et devant un contexte jamais vu, il doit reculer.

## Étape 5 : la perplexité, et le lissage

Comment dire si un modèle est bon ? Pas avec le texte qu’il a appris : un modèle qui recopie son cours le sait par cœur. On le teste sur le texte de validation, qu’il n’a jamais lu. La **perplexité** mesure sa surprise : pour chaque caractère du texte, on regarde la probabilité `p` que le modèle lui donnait, on prend −ln p (0 si le modèle était sûr, très grand s’il jugeait ça presque impossible), on fait la moyenne, puis l’exponentielle. Une perplexité de 12 veut dire que le modèle hésite comme s’il choisissait, à chaque caractère, entre 12 possibilités également probables.

Il y a un piège. Si le modèle n’a jamais vu un caractère après un contexte, son compte est 0, donc sa probabilité aussi, et −ln 0 est infini : la perplexité serait infinie dès la première surprise. Le remède, le **lissage de Laplace**, ajoute 1 au compte de chaque caractère possible, y compris ceux qu’on n’a jamais vus. La probabilité devient (compte + 1) ÷ (total + V), où V est le nombre de caractères différents. Rien n’est plus impossible, et un contexte jamais vu donne la même probabilité à tous les caractères. La fonction `distribution` rend cette probabilité lissée pour tous les caractères, du plus probable au moins probable. Le lissage ne sert qu’à mesurer : pour écrire, on ne tire que parmi les caractères déjà vus.

In [ ]:
from modele_de_langage.ngrammes import distribution

# Après « q » : presque toujours « u », mais rien n’est impossible.
print("Après « q »\u202f:")
for caractere, p in distribution(comptes, lettres, "q", 1)[:3]:
    print(f"  {voir(caractere)}\u202f: probabilité {nombre(p, 3)}")
derniere = distribution(comptes, lettres, "q", 1)[-1]
print(f"Le moins probable après « q »\u202f: {voir(derniere[0])}, probabilité {nombre(derniere[1], 4)}")

# Avec 3 caractères de contexte, beaucoup de groupes n’apparaissent jamais dans le texte.
comptes_3 = comptes_par_ordre[3]
inconnu = "Zq¤"
print(f"Le contexte « {inconnu} » n’est pas dans le texte\u202f: {inconnu not in comptes_3}")
uniformes = {p for _, p in distribution(comptes_3, lettres, inconnu, 3)}
print(f"Après lui, tous les caractères ont la même probabilité, 1 ÷ {len(lettres)} = {nombre(1 / len(lettres), 5)}\u202f: {len(uniformes) == 1}")

In [ ]:
from modele_de_langage.ngrammes import part_jamais_vue, perplexite

sur_entrainement, sur_validation = [], []
for n in range(0, 7):
    comptes_n = compter(entrainement, n)
    sur_entrainement.append(perplexite(comptes_n, lettres, n, entrainement))
    sur_validation.append(perplexite(comptes_n, lettres, n, validation))
    absents, total = part_jamais_vue(comptes_n, validation, n)
    print(
        f"n = {n}\u202f: perplexité {nombre(sur_entrainement[n], 1)} sur le texte appris, "
        f"{nombre(sur_validation[n], 1)} sur le texte jamais lu ; "
        f"contextes jamais vus\u202f: {nombre(100 * absents / total, 1)}\u202f%"
    )

# La phrase ci-dessous est un résultat mesuré : ces vérifications la gardent vraie, ou arrêtent la cellule.
meilleur = sur_validation.index(min(sur_validation))
for nom, valeurs in [("texte appris", sur_entrainement), ("texte jamais lu", sur_validation)]:
    bas = valeurs.index(min(valeurs))
    assert 0 < bas < 6 and valeurs[bas:] == sorted(valeurs[bas:]), f"La perplexité sur le {nom} ne baisse plus puis ne remonte plus : relis le texte de l’étape."
print()
print(f"Sur le texte jamais lu, la perplexité baisse jusqu’à n = {meilleur}, puis remonte.")

La perplexité baisse d’abord quand `n` grandit : plus de contexte aide à deviner. Puis elle remonte, sur le texte appris comme sur le texte jamais lu : au-delà d’un certain `n`, les contextes sont si longs que le modèle ne les a vus qu’une fois ou deux, et le lissage dilue ce qu’il sait. Sur le texte jamais lu, c’est pire : la part des contextes que le modèle n’a jamais vus grandit avec `n`, et il doit alors deviner au hasard. Le meilleur `n` n’est pas le plus grand : il dépend de la quantité de texte à apprendre. Un modèle qui ne retient que des cas particuliers ne généralise pas : c’est le **surapprentissage**. (La cellule précédente vérifie que ce qui est dit ici est vrai pour ce texte.)

## Exercice : le caractère le plus probable

Écris la fonction `le_plus_probable(comptes, contexte, n)`. Elle reçoit les comptes de `compter(texte, n)`, un contexte, et le nombre `n` de caractères de contexte. Elle renvoie le caractère le plus compté après les `n` derniers caractères du contexte. À égalité, elle prend le plus petit point de code, comme le fait `distribution`.

La cellule l’essaie sur trois contextes du texte, et la compare au premier caractère rendu par `distribution`. Si les trois réponses sont les mêmes, le défi est réussi.

Indice : `comptes[contexte]` donne les comptes de ce qui le suit ; `min(suivants, key=…)` choisit le plus petit selon une clé, et la clé `(-suivants[c], c)` range d’abord par compte décroissant, puis par caractère.

In [ ]:
from modele_de_langage.ngrammes import alphabet, charger_texte, compter, couper, distribution

texte = charger_texte()
lettres = alphabet(texte)
n = 3
comptes_3 = compter(couper(texte)[0], n)


def le_plus_probable(comptes, contexte, n):
    # À toi : le caractère le plus compté après les n derniers caractères de contexte.
    return ""


# Ne change rien en dessous : cette partie essaie ta fonction sur trois contextes.
contextes = ["ner ", "ion ", " de "]
reussis = 0
for contexte in contextes:
    voulu = distribution(comptes_3, lettres, contexte, n)[0][0]
    obtenu = le_plus_probable(comptes_3, contexte, n)
    if obtenu == voulu:
        reussis += 1
    else:
        print(f"Après {contexte!r}, ta fonction donne {obtenu!r} au lieu de {voulu!r}.")
if reussis == len(contextes):
    print("Défi réussi\u202f: les trois contextes donnent le bon caractère.")
else:
    print(f"Pas encore ({reussis} contextes sur {len(contextes)}).")

<details>
<summary>Voir une correction (essaie d’abord !)</summary>

`comptes[contexte]` est un compteur : à chaque caractère, combien de fois il suit ce contexte. La clé `(-suivants[c], c)` met le plus compté en premier, et, à égalité, le plus petit point de code. `min` rend le premier.

Remplace la fonction de la cellule de l’exercice par :

```python
def le_plus_probable(comptes, contexte, n):
    suivants = comptes[contexte[len(contexte) - n:]]
    return min(suivants, key=lambda c: (-suivants[c], c))
```

</details>

## Dernière étape : les mêmes textes que la page ?

La page du site n’utilise pas Python : sa figure refait ce modèle en TypeScript, directement dans ton navigateur. Pour être sûr que les deux écrivent exactement les mêmes textes, le dépôt garde un fichier de référence, `tests/attendus.json`, calculé une fois pour toutes par le Python : l’alphabet, les probabilités après des contextes de référence, la part de contextes jamais vus et des suites écrites à chaque température. Les tests de la page relisent ce fichier. La cellule suivante recalcule tout avec les fonctions du projet, et compare au bit près chaque nombre et chaque caractère.

La perplexité passe par l’exponentielle et le logarithme, dont le dernier chiffre peut changer d’un ordinateur à l’autre : la page du site ne la recalcule pas, elle lit les valeurs du fichier, et la cellule les compare à un milliardième près.

In [ ]:
import json
import math

from modele_de_langage.attendus import SECTIONS_EXACTES, SORTIES_LIBM, calculer_attendus, vers_json

attendus = json.loads((projet / "tests" / "attendus.json").read_text(encoding="utf-8"))
calcule = calculer_attendus()

print(
    f"{len(attendus['alphabet'])} caractères, {len(attendus['ordres'])} ordres, "
    f"{len(attendus['suites'])} suites écrites et {len(attendus['cas_limites'])} cas limites comparés."
)
# Chaque section est écrite en JSON, comme dans le fichier, puis comparée caractère par
# caractère : « == » prendrait -0.0 pour 0.0, et 6 pour 6.0.
differentes = [nom for nom in SECTIONS_EXACTES if vers_json(calcule[nom]) != vers_json(attendus[nom])]


def sans_perplexites(ordres):
    return [{cle: v for cle, v in ordre.items() if cle not in SORTIES_LIBM} for ordre in ordres]


# Les ordres : tout au bit près, sauf les perplexités.
if vers_json(sans_perplexites(calcule["ordres"])) != vers_json(sans_perplexites(attendus["ordres"])):
    differentes.append("ordres")
for fait, lu in zip(calcule["ordres"], attendus["ordres"]):
    for cle in SORTIES_LIBM:
        if not math.isclose(fait[cle], lu[cle], rel_tol=1e-9):
            differentes.append(f"ordres, n = {lu['n']}, {cle}")
for nom in differentes:
    print(f"La section «\u202f{nom}\u202f» n’est pas celle du fichier de référence.")

print(f"Même résultat que la page du site\u202f: {'oui' if not differentes else 'non'}")  # \u202f est l’espace fine insécable

## Pour aller plus loin

- Le code complet du projet, avec ses tests : [projets/1.1-modele-de-langage sur GitHub](https://github.com/chipat-neko/forge-ia/tree/main/projets/1.1-modele-de-langage).
- La page du chapitre, pour écrire avec le modèle, changer la température et lire le code expliqué : [chapitre 1.1](https://chipat-neko.github.io/forge-ia/partie-1/1-1/).
- Depuis le dossier du projet, `uv run python -m modele_de_langage suite --n 4 --temperature 1 --graine 0 "Le forgeron "` écrit une suite sur ton ordinateur.

Le code de ce carnet est sous licence MIT, ses textes sous licence CC BY 4.0.